In [2]:
!pip install accelerate bitsandbytes sentencepiece torch seqeval

  Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-win_amd64.whl.metadata (10 kB)
  Using cached seqeval-1.2.2-py3-none-any.whl
Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl (55.4 MB)
Using cached sentencepiece-0.2.1-cp312-cp312-win_amd64.whl (1.1 MB)

   ---------------------------------------- 0/4 [sentencepiece]
   ---------------------------------------- 0/4 [sentencepiece]
   ---------------------------------------- 0/4 [sentencepiece]
   ---------- ----------------------------- 1/4 [seqeval]
   ---------- ----------------------------- 1/4 [seqeval]
   ---------- ----------------------------- 1/4 [seqeval]
   -------------------- ------------------- 2/4 [bitsandbytes]
   -------------------- ------------------- 2/4 [bitsandbytes]
   -------------------- ------------------- 2/4 [bitsandbytes]
   -------------------- ------------------- 2/4 [bitsandbytes]
   -------------------- ------------------- 2/4 

In [3]:
import json
import re
import random
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from enum import Enum
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from seqeval.metrics import f1_score, precision_score, recall_score

# --- Device & Seed ---
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("Используем CPU")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print("✅ Инициализация завершена")

BASE_DIR      = Path("E:/Code/Poliforge/Polyforge-AI/Polyforge-AI")
DATA_RAW_DIR  = BASE_DIR / "data" / "raw"
DATA_PROC_DIR = BASE_DIR / "data" / "processed"
DATA_CFG_DIR  = BASE_DIR / "data" / "configs"
MODELS_DIR    = BASE_DIR / "models" / "nlp_model_bert"
REPORTS_DIR   = MODELS_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

Используем CPU
Device: cpu
✅ Инициализация завершена


In [3]:
import copy

# ============================================================
# БЛОК 2: Загрузка и ОСТОРОЖНАЯ аугментация
# ============================================================
# Загружаем расширенный датасет
with open(DATA_CFG_DIR / "natural_requests_350.json", encoding="utf-8") as f:
    FULL_DATASET = json.load(f)

print(f"Загружено оригинальных запросов: {len(FULL_DATASET)}")

# --- Безопасные преобразования ---
PREFIXES = [
    "", "", "",
    "Подскажите, ", "Посоветуйте, ", "Нужен совет: ",
    "Интересует: ", "Задача: ", "Мне бы хотелось: "
]
SUFFIXES = [
    "", "", "",
    " Спасибо.", " Заранее благодарю.", " Буду признателен.",
    " Жду предложений.", " Это срочно.", " Можно аналоги."
]

# Синонимы единиц измерения (для замены в VALUE)
UNIT_SYNONYMS = {
    "°C": ["°C", "градусов Цельсия", "C"],
    "ГПа": ["ГПа", "гигапаскаль"],
    "МПа": ["МПа", "мегапаскаль", "MPa"],
    "г/см³": ["г/см³", "г/см3", "грамм на кубический сантиметр"],
    "кВ/мм": ["кВ/мм", "киловольт на миллиметр"],
    "Баррер": ["Баррер", "Barrer"],
    "Дж/(г·К)": ["Дж/(г·К)", "J/(g·K)"],
    "%": ["%", "процентов"],
    "": [""]
}

def insert_prefix_suffix(text, entities, prefix, suffix):
    new_text = prefix + text + suffix
    offset = len(prefix)
    new_entities = [{**e, "start": e["start"]+offset, "end": e["end"]+offset} for e in entities]
    return new_text, new_entities

def replace_unit_in_value(text: str, entities: list):
    """Заменяет единицу измерения в VALUE-сущностях на синоним."""
    value_ents = [e for e in entities if e["label"] == "VALUE"]
    if not value_ents:
        return text, entities
    new_text = text
    new_entities = copy.deepcopy(entities)
    shift = 0
    for ent in sorted(value_ents, key=lambda x: x["start"]):
        original_val = text[ent["start"]:ent["end"]]
        # Ищем единицу в конце значения (простая эвристика)
        for unit, syns in UNIT_SYNONYMS.items():
            if unit and original_val.endswith(unit):
                new_unit = random.choice(syns)
                if new_unit != unit:
                    new_val = original_val[:-len(unit)] + new_unit
                    start = ent["start"] + shift
                    end = ent["end"] + shift
                    new_text = new_text[:start] + new_val + new_text[end:]
                    ent["end"] = start + len(new_val)
                    shift += len(new_val) - len(original_val)
                break
    # Корректируем остальные сущности
    for ent2 in new_entities:
        if ent2["start"] > ent["start"]:
            ent2["start"] += shift
            ent2["end"] += shift
    return new_text, new_entities

def shuffle_entities_in_text(text, entities):
    """Перестановка двух блоков свойств (как раньше)"""
    # ... код остаётся прежним ...
    # (оставляем вашу реализацию, она корректна)

def augment_safe(sample):
    """Создаёт 3-5 безопасных вариантов."""
    variants = [sample]  # оригинал
    text = sample["text"]
    entities = sample["entities"]

    # Префиксы/суффиксы (2 варианта)
    for _ in range(2):
        prefix = random.choice(PREFIXES)
        suffix = random.choice(SUFFIXES)
        new_text, new_ents = insert_prefix_suffix(text, entities, prefix, suffix)
        variants.append({"text": new_text, "entities": new_ents})

    # Замена единиц (1 вариант)
    new_text, new_ents = replace_unit_in_value(text, entities)
    if new_text != text:
        variants.append({"text": new_text, "entities": new_ents})

    # Перестановка свойств (если >2 свойств, 1 вариант)
    if len(entities) > 2:
        new_text, new_ents = shuffle_entities_in_text(text, entities)
        variants.append({"text": new_text, "entities": new_ents})

    return variants

def shuffle_entities_in_text(text, entities):
    """Переставляет два случайных блока с сущностями, разделённых запятой/союзом."""
    parts = re.split(r'(, и |, а также |, | и |; | \+)', text)
    if len(parts) < 3:
        return text, entities

    fragment_indices = []
    pos = 0
    for i, part in enumerate(parts):
        fragment_indices.append((i, pos, pos+len(part), part))
        pos += len(part)

    entity_fragments = []
    for i, s, e, p in fragment_indices:
        ents_in = [ent for ent in entities if ent["start"] >= s and ent["end"] <= e]
        if ents_in:
            entity_fragments.append((i, s, e, p, ents_in))

    if len(entity_fragments) >= 2:
        a, b = random.sample(range(len(entity_fragments)), 2)
        fa, fb = entity_fragments[a], entity_fragments[b]
        new_parts = list(parts)
        new_parts[fa[0]] = fb[3]
        new_parts[fb[0]] = fa[3]
        new_text = "".join(new_parts)

        offset_map = {}
        offset_map[fa[0]] = fb[2] - fa[1]
        offset_map[fb[0]] = fa[2] - fb[1]
        new_entities = []
        for ent in entities:
            new_ent = ent.copy()
            for fidx, s, e, _, _ in entity_fragments:
                if ent["start"] >= s and ent["end"] <= e:
                    if fidx in offset_map:
                        new_ent["start"] += offset_map[fidx]
                        new_ent["end"] += offset_map[fidx]
                    break
            new_entities.append(new_ent)
        return new_text, new_entities
    return text, entities

# Разделение 70/15/15
random.seed(SEED)
shuffled = random.sample(FULL_DATASET, len(FULL_DATASET))
n_total = len(shuffled)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)

train_orig = shuffled[:n_train]
val_orig   = shuffled[n_train:n_train + n_val]
test_orig  = shuffled[n_train + n_val:]

# Аугментация только для train
train_augmented = []
for sample in train_orig:
    train_augmented.extend(augment_safe(sample))

# Val и test без аугментации
val_real = val_orig
test_real = test_orig

print(f"Train (безопасная аугментация): {len(train_augmented)} примеров")
print(f"Val (оригинальные):              {len(val_real)} примеров")
print(f"Test (оригинальные):             {len(test_real)} примеров")

Загружено оригинальных запросов: 412
Train (безопасная аугментация): 1094 примеров
Val (оригинальные):              61 примеров
Test (оригинальные):             63 примеров


In [5]:
# ============================================================
# Блок 2.1: Back-translation + безопасная аугментация
# ============================================================
from transformers import MarianMTModel, MarianTokenizer
from difflib import SequenceMatcher

# Загружаем модели перевода (уже скачаны)
mt_ru_en = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-ru-en').to(DEVICE)
tok_ru_en = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-ru-en')
mt_en_ru = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-ru').to(DEVICE)
tok_en_ru = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-ru')

def back_translate(text: str) -> str:
    """RU → EN → RU"""
    # RU → EN
    inputs = tok_ru_en(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    translated = mt_ru_en.generate(**inputs, max_new_tokens=256)
    en_text = tok_ru_en.batch_decode(translated, skip_special_tokens=True)[0]
    # EN → RU
    inputs = tok_en_ru(en_text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    translated = mt_en_ru.generate(**inputs, max_new_tokens=256)
    ru_text = tok_en_ru.batch_decode(translated, skip_special_tokens=True)[0]
    return ru_text

def align_entities(original_text: str, new_text: str, entities: list) -> list:
    """Переносит координаты сущностей с оригинала на новый текст."""
    matcher = SequenceMatcher(None, original_text, new_text)
    new_entities = []
    for ent in entities:
        orig_start, orig_end = ent["start"], ent["end"]
        best_start, best_end = None, None
        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            if tag == 'equal' and i1 <= orig_start < i2 and i1 < orig_end <= i2:
                offset = j1 - i1
                best_start = orig_start + offset
                best_end = orig_end + offset
                break
        if best_start is not None:
            new_entities.append({**ent, "start": best_start, "end": best_end})
        else:
            # fallback – поиск подстроки
            fragment = original_text[orig_start:orig_end]
            idx = new_text.find(fragment)
            if idx != -1:
                new_entities.append({**ent, "start": idx, "end": idx + len(fragment)})
    return new_entities

def augment_with_backtranslation(sample: dict) -> list:
    """Генерирует варианты: оригинал + back-translation + augment_safe."""
    variants = [sample]                     # оригинал
    text, entities = sample["text"], sample["entities"]

    # 1. Back-translation (1 вариант)
    try:
        new_text = back_translate(text)
        new_ents = align_entities(text, new_text, entities)
        variants.append({"text": new_text, "entities": new_ents})
    except Exception as e:
        print(f"Back-translation failed: {e}")

    # 2. Безопасная аугментация (3-5 вариантов)
    variants.extend(augment_safe(sample)[1:])   # исключаем оригинал

    return variants

print("✅ Блок 2.1 готов")

✅ Блок 2.1 готов


In [6]:
# ============================================================
# Блок 2.2: Генерация расширенного датасета (с кэшированием)
# ============================================================
CACHE_TRAIN = DATA_PROC_DIR / "train_backtrans_augmented.pkl"

if CACHE_TRAIN.exists():
    with open(CACHE_TRAIN, "rb") as f:
        train_augmented = pickle.load(f)
    print(f"Загружено {len(train_augmented)} примеров из кэша")
else:
    train_augmented = []
    for sample in tqdm(train_orig, desc="Аугментация (back-translation + safe)"):
        variants = augment_with_backtranslation(sample)
        train_augmented.extend(variants)
    with open(CACHE_TRAIN, "wb") as f:
        pickle.dump(train_augmented, f)
    print(f"Сгенерировано {len(train_augmented)} примеров")

# Валидация и тест – оригинальные
val_real = val_orig
test_real = test_orig

print(f"Train (расширенный): {len(train_augmented)}")
print(f"Val (оригинальные):  {len(val_real)}")
print(f"Test (оригинальные): {len(test_real)}")

Аугментация (back-translation + safe):   0%|          | 0/288 [00:00<?, ?it/s]

Сгенерировано 1376 примеров
Train (расширенный): 1376
Val (оригинальные):  61
Test (оригинальные): 63


In [7]:
from transformers import AutoConfig

# ============================================================
# БЛОК 3: Токенизатор и BIO-разметка (rubert-tiny2)
# ============================================================
MODEL_NAME = "cointegrated/rubert-tiny2"  # 29M параметров, меньше переобучение
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

LABEL2ID = {
    "O":           0,
    "B-PROPERTY":  1,
    "I-PROPERTY":  2,
    "B-VALUE":     3,
    "I-VALUE":     4,
    "B-QUALIFIER": 5,
    "I-QUALIFIER": 6,
}
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = NUM_LABELS
config.hidden_dropout_prob = 0.1          # уменьшенный dropout
config.attention_probs_dropout_prob = 0.1
config.id2label = ID2LABEL
config.label2id = LABEL2ID

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)

# Заморозка первых 6 слоёв энкодера (опционально, если будет переобучение)
# for name, param in model.bert.encoder.layer.named_parameters():
#     layer_num = int(name.split(".")[0])
#     if layer_num < 6:
#         param.requires_grad = False

print(f"Параметров: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Вспомогательные функции разметки (без изменений)
def annotate_bio(text: str, entities: list) -> list:
    """Токенизирует текст по словам и присваивает BIO-метки."""
    tokens = []
    for m in re.finditer(r'\w+|[^\w\s]', text):
        tokens.append({
            "text":  m.group(),
            "start": m.start(),
            "end":   m.end(),
            "label": "O",
        })
    for ent in entities:
        e_start = ent.get("start", -1)
        e_end   = ent.get("end",   -1)
        label   = ent.get("label", "")
        if e_start == -1 or e_end == -1 or label not in ("PROPERTY", "VALUE", "QUALIFIER"):
            continue
        first = True
        for tok in tokens:
            if tok["start"] >= e_start and tok["end"] <= e_end:
                tok["label"] = f"B-{label}" if first else f"I-{label}"
                first = False
    return [(t["text"], t["label"]) for t in tokens]

def sample_to_features(sample: dict, tokenizer, max_length: int = 256) -> dict | None:
    text     = sample.get("text", "")
    entities = sample.get("entities", [])
    if not text.strip():
        return None
    pairs = annotate_bio(text, entities)
    if not pairs:
        return None
    words, labels = zip(*pairs)
    encoding = tokenizer(
        list(words),
        is_split_into_words=True,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    word_ids = encoding.word_ids(batch_index=0)
    aligned_labels = []
    prev_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
        elif word_id != prev_word_id:
            label_id = LABEL2ID.get(labels[word_id], 0)
            aligned_labels.append(label_id)
        else:
            aligned_labels.append(-100)
        prev_word_id = word_id
    aligned_labels = [l if l in range(NUM_LABELS) or l == -100 else 0 for l in aligned_labels]
    labels_tensor = torch.tensor(aligned_labels, dtype=torch.long)
    return {
        "input_ids":      encoding["input_ids"].squeeze(),
        "attention_mask": encoding["attention_mask"].squeeze(),
        "labels":         labels_tensor,
    }

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Параметров: 29.1M


In [8]:
# ============================================================
# БЛОК 4: Dataset и DataLoader (BATCH_SIZE=16, max_length=128)
# ============================================================
class PolymerNERDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=128):
        self.features = []
        skipped = 0
        for s in tqdm(samples, desc="Токенизация"):
            f = sample_to_features(s, tokenizer, max_length)
            if f is not None:
                self.features.append(f)
            else:
                skipped += 1
        print(f"  Создано примеров: {len(self.features)} (пропущено: {skipped})")

    def __len__(self): return len(self.features)
    def __getitem__(self, i): return self.features[i]

def collate_fn(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "labels":         torch.stack([b["labels"] for b in batch]),
    }

MAX_LENGTH = 128
BATCH_SIZE = 16
LR = 1e-5
EPOCHS = 15
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
PATIENCE = 5
GRAD_CLIP = 1.0

train_dataset = PolymerNERDataset(train_augmented, tokenizer, MAX_LENGTH)
val_dataset   = PolymerNERDataset(val_real, tokenizer, MAX_LENGTH)
test_dataset  = PolymerNERDataset(test_real, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train батчей: {len(train_loader)} | Val батчей: {len(val_loader)} | Test батчей: {len(test_loader)}")
print("✅ Блок 4 готов")

Токенизация:   0%|          | 0/1376 [00:00<?, ?it/s]

  Создано примеров: 1376 (пропущено: 0)


Токенизация:   0%|          | 0/61 [00:00<?, ?it/s]

  Создано примеров: 61 (пропущено: 0)


Токенизация:   0%|          | 0/63 [00:00<?, ?it/s]

  Создано примеров: 63 (пропущено: 0)
Train батчей: 86 | Val батчей: 4 | Test батчей: 4
✅ Блок 4 готов


In [11]:
# ============================================================
# БЛОК 5: Обучение (исправлены ошибки seqeval и label_smoothing)
# ============================================================
# 1. Функция потерь (без label_smoothing для совместимости)
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# 2. Оптимизатор и планировщик (с проверкой типов)
no_decay = ["bias", "LayerNorm.weight"]
optimizer = AdamW([
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay) and p.requires_grad],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay) and p.requires_grad],
     "weight_decay": 0.0},
], lr=LR)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# 3. Вспомогательные функции
def decode_predictions(logits, labels):
    preds = logits.argmax(dim=-1).cpu().numpy()
    labs  = labels.cpu().numpy()
    preds_list, labels_list = [], []
    for pred_seq, lab_seq in zip(preds, labs):
        p_row, l_row = [], []
        for p, l in zip(pred_seq, lab_seq):
            if l == -100: continue
            p_row.append(ID2LABEL[p])
            l_row.append(ID2LABEL[l])
        preds_list.append(p_row)
        labels_list.append(l_row)
    return preds_list, labels_list

def evaluate_epoch(model, loader, desc="Val"):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, leave=False):
            out = model(
                input_ids      = batch["input_ids"].to(DEVICE),
                attention_mask = batch["attention_mask"].to(DEVICE),
                labels         = batch["labels"].to(DEVICE),
            )
            logits = out.logits
            loss = criterion(logits.permute(0,2,1), batch["labels"].to(DEVICE))
            total_loss += loss.item()
            p, l = decode_predictions(out.logits, batch["labels"])
            all_preds.extend(p)
            all_labels.extend(l)
    avg_loss = total_loss / max(len(loader), 1)
    # seqeval по умолчанию использует IOB2 (аналог BIO), scheme не передаём
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    p  = precision_score(all_labels, all_preds, zero_division=0)
    r  = recall_score(all_labels, all_preds, zero_division=0)
    return avg_loss, f1, p, r

# 4. Цикл обучения с ранней остановкой по val F1
history = {"train_loss": [], "val_loss": [], "val_f1": [], "val_p": [], "val_r": []}
best_f1 = 0.0
best_state = None
no_improve = 0

print(f"\n{'='*60}")
print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>9} | "
      f"{'F1':>6} | {'P':>6} | {'R':>6}")
print(f"{'='*60}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        optimizer.zero_grad()
        out = model(
            input_ids      = batch["input_ids"].to(DEVICE),
            attention_mask = batch["attention_mask"].to(DEVICE),
            labels         = batch["labels"].to(DEVICE),
        )
        logits = out.logits
        loss = criterion(logits.permute(0,2,1), batch["labels"].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss, val_f1, val_p, val_r = evaluate_epoch(model, val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)
    history["val_p"].append(val_p)
    history["val_r"].append(val_r)

    print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>9.4f} | "
          f"{val_f1:>6.3f} | {val_p:>6.3f} | {val_r:>6.3f}")

    if val_f1 > best_f1 + 1e-4:
        best_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
        print(f"        ↑ новый лучший F1={best_f1:.4f}")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\n⏹  Early stopping на эпохе {epoch}")
            break

if best_state:
    model.load_state_dict(best_state)
    print(f"\n✅ Лучшая модель загружена. F1={best_f1:.4f}")


 Epoch | Train Loss |  Val Loss |     F1 |      P |      R


Epoch 1/15:   0%|          | 0/86 [00:00<?, ?it/s]

Val:   0%|          | 0/4 [00:00<?, ?it/s]

     1 |     1.7811 |    1.4388 |  0.000 |  0.000 |  0.000


Epoch 2/15:   0%|          | 0/86 [00:00<?, ?it/s]

...

Val:   0%|          | 0/4 [00:00<?, ?it/s]

    14 |     0.6650 |    0.8288 |  0.165 |  0.290 |  0.115


Epoch 15/15:   0%|          | 0/86 [00:00<?, ?it/s]

Val:   0%|          | 0/4 [00:00<?, ?it/s]

    15 |     0.6637 |    0.8280 |  0.165 |  0.290 |  0.115

✅ Лучшая модель загружена. F1=0.1651


In [ ]:
# ============================================================
# БЛОК 5.5: Детальная оценка на тестовом наборе + отчёты
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.metrics import confusion_matrix
from seqeval.metrics import classification_report

# Финальная оценка на test_loader
print("="*60)
print("ФИНАЛЬНАЯ ОЦЕНКА НА ТЕСТОВОМ НАБОРЕ")
print("="*60)
test_loss, test_f1, test_p, test_r = evaluate_epoch(model, test_loader, "Test")
print(f"\n📊 ТЕСТОВЫЕ МЕТРИКИ:")
print(f"  Loss:      {test_loss:.4f}")
print(f"  F1:        {test_f1:.4f}")
print(f"  Precision: {test_p:.4f}")
print(f"  Recall:    {test_r:.4f}")

# Собираем предсказания для confusion matrix
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test predictions"):
        out = model(
            input_ids      = batch["input_ids"].to(DEVICE),
            attention_mask = batch["attention_mask"].to(DEVICE),
            labels         = batch["labels"].to(DEVICE),
        )
        p, l = decode_predictions(out.logits, batch["labels"])
        all_preds.extend(p)
        all_labels.extend(l)

# Classification report
print("\n📋 CLASSIFICATION REPORT (тест):")
report_text = classification_report(all_labels, all_preds, digits=4)
print(report_text)
with open(REPORTS_DIR / "test_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

# Confusion matrix
y_true_flat = [l for seq in all_labels for l in seq]
y_pred_flat = [p for seq in all_preds  for p in seq]
label_names = list(LABEL2ID.keys())

cm = confusion_matrix(y_true_flat, y_pred_flat, labels=label_names)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names,
            ax=axes[0], cbar_kws={"label": "Count"})
axes[0].set_title("Confusion Matrix (counts)")

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Greens",
            xticklabels=label_names, yticklabels=label_names,
            ax=axes[1], cbar_kws={"label": "Recall"})
axes[1].set_title("Confusion Matrix (normalized)")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "test_confusion.png", dpi=150)
plt.show()

# График обучения (как раньше) – используйте history
print("✅ Отчёты сохранены в", REPORTS_DIR)

In [ ]:
# ============================================================
# БЛОК 9: Сохранение модели и артефактов
# ============================================================
import matplotlib.pyplot as plt

# График
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, len(history["train_loss"]) + 1)

ax1.plot(ep, history["train_loss"], "b-o", markersize=4, label="Train Loss")
ax1.plot(ep, history["val_loss"],   "r-o", markersize=4, label="Val Loss")
ax1.set_title("Loss"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, history["val_f1"], "g-o", markersize=4, label="Val F1")
ax2.plot(ep, history["val_p"],  "b--", markersize=3, label="Val P")
ax2.plot(ep, history["val_r"],  "r--", markersize=3, label="Val R")
ax2.axhline(y=best_f1, color="g", linestyle=":", alpha=0.5)
ax2.set_title("F1 / P / R"); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(MODELS_DIR / "training_curves.png", dpi=150)
plt.show()

# Сохраняем модель
model.save_pretrained(MODELS_DIR)
tokenizer.save_pretrained(MODELS_DIR)

# Артефакты
artifacts = {
    "label2id":         LABEL2ID,
    "id2label":         ID2LABEL,
    "model_name":       MODEL_NAME,
    "best_val_f1":      best_f1,
    "training_history": history,
}

artifacts_path = MODELS_DIR / "artifacts.pkl"
with open(artifacts_path, "wb") as f:
    pickle.dump(artifacts, f, protocol=4)

# Проверка
with open(artifacts_path, "rb") as f:
    test = pickle.load(f)
assert "thresholds" in test
print(f"✅ artifacts.pkl: {artifacts_path.stat().st_size} байт")

print(f"✅ Модель сохранена в {MODELS_DIR}")
print(f"   Best Val F1: {best_f1:.4f}")